In [8]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

In [9]:
df = pd.read_excel("Concrete_Data.xls")

X = df.iloc[:, :-1]   # all columns except the last one
y = df.iloc[:, -1]    # the last column (strength)


def categorize_strength(value):
    if value < 25.01:
        return 0  # low
    elif value < 55.02:
        return 1  # medium
    else:
        return 2  # high

y_classes = np.array([categorize_strength(val) for val in y])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_classes, test_size=0.2, random_state=42, stratify=y_classes
)

scaler = StandardScaler()
X_train_scaled_withoutCV = scaler.fit_transform(X_train)
X_test_scaled_withoutCV = scaler.transform(X_test)

# 1. Baseline Model

The baseline will be a model which compute the largest class on the training data, and predict everything in the
test-data as belonging to that class (corresponding to the optimal prediction by a logistic
regression model with a bias term and no features

In [10]:
num_low = np.sum(y_classes == 0)
num_medium = np.sum(y_classes == 1)
num_high = np.sum(y_classes == 2)

print(f"Class distribution:\n Low: {num_low}\n Medium: {num_medium}\n High: {num_high}")

Class distribution:
 Low: 295
 Medium: 588
 High: 147


In [11]:
# sicne the medium class has the most samples, the baseline model will predict this class everytime 

from sklearn.dummy import DummyClassifier
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_scaled_withoutCV, y_train)
y_pred_base = baseline.predict(X_test_scaled_withoutCV)

print(confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base))   

[[  0  59   0]
 [  0 118   0]
 [  0  29   0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        59
           1       0.57      1.00      0.73       118
           2       0.00      0.00      0.00        29

    accuracy                           0.57       206
   macro avg       0.19      0.33      0.24       206
weighted avg       0.33      0.57      0.42       206



/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/theresabartels/miniforge3/envs/dtu02452/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [17]:
print(y_pred_base)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


# 2. ANN

In [12]:
from sklearn.neural_network import MLPClassifier

model_mlp = MLPClassifier(hidden_layer_sizes=(64, 32, 16), 
                      activation='relu', solver='adam',
                      #learning_rate='adaptive',
                      #learning_rate_init=0.005,
                      max_iter=3000,
                      random_state=42)


model_mlp.fit(X_train_scaled_withoutCV, y_train)
predictions_ann = model_mlp.predict(X_test_scaled_withoutCV)

test with normal train/test split

In [13]:
print(confusion_matrix(y_test, predictions_ann))
print(classification_report(y_test, predictions_ann))   

[[ 51   8   0]
 [  8 108   2]
 [  0   4  25]]
              precision    recall  f1-score   support

           0       0.86      0.86      0.86        59
           1       0.90      0.92      0.91       118
           2       0.93      0.86      0.89        29

    accuracy                           0.89       206
   macro avg       0.90      0.88      0.89       206
weighted avg       0.89      0.89      0.89       206



two level cross validation: 

In [14]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

results = []
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

# logistic regression model
logreg = LogisticRegression(max_iter=500)
logreg_params = {'C': [0.01, 0.1, 1, 10]}

#grid search: parameters than can be chosen
mlp_params = {
    'hidden_layer_sizes': [(64,), (64, 32), (64, 32, 16)],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.005],
    'learning_rate': ['constant', 'adaptive']
}

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y_classes[train_idx], y_classes[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Baseline
    baseline.fit(X_train_scaled, y_train)
    y_pred_base = baseline.predict(X_test_scaled)
    err_base = 1 - accuracy_score(y_test, y_pred_base)

    # logistic regression
    grid_logreg = GridSearchCV(logreg, logreg_params, cv=inner_cv)
    grid_logreg.fit(X_train_scaled, y_train)
    y_pred_logreg = grid_logreg.predict(X_test_scaled)
    err_logreg = 1 - accuracy_score(y_test, y_pred_logreg)

    # MLP
    grid_mlp = GridSearchCV(model_mlp, mlp_params, cv=inner_cv)
    grid_mlp.fit(X_train_scaled, y_train)
    y_pred_mlp = grid_mlp.predict(X_test_scaled)
    err_mlp = 1 - accuracy_score(y_test, y_pred_mlp)

    # Ergebnisse speichern
    results.append({
        'Baseline Error': err_base,
        'LogReg Error': err_logreg,
        'LogReg Params': grid_logreg.best_params_,
        'MLP Error': err_mlp,
        'MLP Params': grid_mlp.best_params_,
    })


In [19]:
df_results = pd.DataFrame(results)

# Best fold for each
best_mlp = df_results.loc[df_results['MLP Error'].idxmin()]
best_logreg = df_results.loc[df_results['LogReg Error'].idxmin()]

print("\nBest MLP fold:\n", best_mlp)
print("\nBest Logistic Regression fold:\n", best_logreg)



Best MLP fold:
 Baseline Error                                             0.504854
LogReg Error                                               0.194175
LogReg Params                                             {'C': 10}
MLP Error                                                  0.116505
MLP Params        {'activation': 'relu', 'hidden_layer_sizes': (...
Name: 1, dtype: object

Best Logistic Regression fold:
 Baseline Error                                             0.402913
LogReg Error                                               0.194175
LogReg Params                                             {'C': 10}
MLP Error                                                  0.135922
MLP Params        {'activation': 'relu', 'hidden_layer_sizes': (...
Name: 0, dtype: object
